# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [10]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_full/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [11]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_full/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_full/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_full/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [12]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                       base             base_inv                       ft   
0           .Today (0.0250)      urrenc (0.0221)          .Today (0.0220)   
1          .Second (0.0111)       act (5.10e-03)         .Second (0.0104)   
2        Buccane (8.67e-03)       pos (4.94e-03)       Buccane (8.61e-03)   
3          /Area (6.32e-03)    askell (3.49e-03)         /Area (6.68e-03)   
4            .au (4.91e-03)      gons (3.30e-03)           .au (3.94e-03)   
5      /problems (3.83e-03)      ejec (2.00e-03)     /problems (3.69e-03)   
6            aru (3.83e-03)        �� (2.00e-03)         /Math (3.36e-03)   
7      /entities (2.90e-03)        دي (2.00e-03)         /bind (3.16e-03)   
8          /Math (2.72e-03)      azon (1.93e-03)     /entities (2.87e-03)   
9       /problem (2.64e-03)     fácil (1.82e-03)          oire (2.70e-03)   
10         /bind (2.56e-03)      anth (1.76e-03)      /problem (2.70e-03)   
11      /respond (2.56e-03)     posix (1.71e-03)     /activity (2.62e-03)   
12          fter (2.47e-03)  essional (1.66e-03)      /respond (2.38e-03)   
13    confidence (2.26e-03)  Optional (1.56e-03)          fter (2.30e-03)   
14     /operator (2.18e-03)      Vers (1.50e-03)   persistence (2.30e-03)   
15     /activity (2.18e-03)        vs (1.46e-03)          ilot (2.17e-03)   
16   persistence (2.12e-03)    Phones (1.46e-03)        soever (2.17e-03)   
17          ilot (2.06e-03)       med (1.25e-03)    confidence (2.17e-03)   
18     belonging (1.98e-03)  >Welcome (1.25e-03)     /operator (2.11e-03)   
19     .AddRange (1.87e-03)      orst (1.25e-03)           aru (2.04e-03)   

                                                                     \
                 ft_inv                   diff             diff_inv   
0       urrenc (0.0143)      gorith (9.77e-03)     ainer (7.93e-03)   
1        pos (5.28e-03)       encia (7.17e-03)   passing (6.20e-03)   
2        act (4.82e-03)        onta (6.71e-03)   Passage (6.01e-03)   
3     askell (4.82e-03)     trovare (5.58e-03)      htub (5.65e-03)   
4         �� (3.20e-03)   -scalable (4.09e-03)       buz (5.13e-03)   
5       gons (3.20e-03)          }? (4.09e-03)       muc (4.67e-03)   
6      fácil (2.82e-03)   acimiento (3.60e-03)   Passing (4.00e-03)   
7         دي (2.14e-03)     Masters (3.17e-03)     arten (3.88e-03)   
8       anth (2.14e-03)    Accounts (2.81e-03)      Pass (3.88e-03)   
9       azon (1.95e-03)      stride (2.81e-03)      prox (3.02e-03)   
10  essional (1.95e-03)         sab (2.81e-03)       ans (3.02e-03)   
11       med (1.95e-03)        .;\n (2.64e-03)      auen (2.93e-03)   
12        vs (1.66e-03)       abbix (2.47e-03)     -town (2.67e-03)   
13    Phones (1.61e-03)  Executable (2.47e-03)       ush (2.67e-03)   
14      ejec (1.51e-03)       guard (2.47e-03)       cre (2.50e-03)   
15  Optional (1.46e-03)        ucci (2.32e-03)     alary (2.50e-03)   
16         د (1.34e-03)        (SQL (2.18e-03)       ubb (2.35e-03)   
17     cambi (1.30e-03)      resden (2.18e-03)      PASS (2.27e-03)   
18       div (1.21e-03)          si (2.18e-03)    endale (2.27e-03)   
19       ')" (1.14e-03)        atra (2.04e-03)         ư (2.27e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9141)          zoek (0.8555)           To (0.9141)   
1           The (0.0454)      contador (0.1309)          The (0.0400)   
2           Let (0.0139)           메 (9.46e-03)           In (0.0157)   
3            In (0.0130)         иск (3.49e-03)          Let (0.0122)   
4         ### (3.72e-03)     Produto (2.12e-03)          A (3.49e-03)   
5           A (2.56e-03)           � (1.42e-05)        ### (3.30e-03)   
6         For (1.14e-03)     Detalle (9.78e-06)        For (1.88e-03)   
7          As (9.16e-04)      Resets (9.78e-06)         ** (1.46e-03)   
8           I (7.82e-04)        

In [13]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.72e-03)        /problem (0.0400)   
1        /entities (0.0273)       (us (5.31e-03)       /entities (0.0266)   
2        /problems (0.0177)      sagt (4.15e-03)       /problems (0.0183)   
3           .Today (0.0101)       ]]; (4.15e-03)       /global (8.91e-03)   
4        /global (8.85e-03)        że (3.65e-03)        .Today (8.61e-03)   
5           /job (7.60e-03)    testim (3.02e-03)          /job (6.71e-03)   
6        /manage (7.60e-03)       ')" (2.84e-03)       /manage (6.71e-03)   
7   /preferences (6.29e-03)       ($. (2.84e-03)       /layout (5.74e-03)   
8        /layout (5.74e-03)      -ves (2.84e-03)  /preferences (5.25e-03)   
9      /provider (5.07e-03)     zeigt (2.67e-03)     /provider (5.07e-03)   
10       /crypto (4.91e-03)     feliz (2.21e-03)       /crypto (4.76e-03)   
11   /connection (4.33e-03)     spons (2.21e-03)   /connection (4.21e-03)   
12      /logging (4.06e-03)     lesen (2.08e-03)      /logging (4.09e-03)   
13    WHATSOEVER (3.94e-03)       (!! (1.95e-03)  /environment (3.83e-03)   
14       /engine (3.94e-03)   kontrol (1.95e-03)       /engine (3.83e-03)   
15          /reg (3.81e-03)      helf (1.72e-03)          /reg (3.71e-03)   
16  /environment (3.81e-03)    spiele (1.72e-03)    WHATSOEVER (3.60e-03)   
17       /dialog (3.48e-03)     scrut (1.62e-03)      /effects (3.60e-03)   
18       /entity (3.37e-03)       fas (1.53e-03)       /dialog (3.39e-03)   
19      /effects (3.37e-03)       )": (1.53e-03)       /object (3.17e-03)   

                                                                     \
                 ft_inv                  diff              diff_inv   
0        .vn (8.36e-03)         ucci (0.0342)         ropa (0.0339)   
1        (us (4.46e-03)       anno (9.77e-03)         ー� (6.26e-03)   
2        ]]; (4.18e-03)       Pert (8.12e-03)      vidéo (5.52e-03)   
3       sagt (3.94e-03)       heim (8.12e-03)      atter (4.58e-03)   
4         że (3.94e-03)      annon (5.95e-03)        vod (4.58e-03)   
5     testim (3.07e-03)       acea (4.91e-03)  versation (4.18e-03)   
6      zeigt (3.07e-03)        lij (4.61e-03)     .spark (4.18e-03)   
7       -ves (3.07e-03)      avras (3.39e-03)   narrowly (3.57e-03)   
8        ($. (2.88e-03)        bel (3.39e-03)        zie (3.57e-03)   
9        ')" (2.88e-03)   independ (3.08e-03)       yles (3.16e-03)   
10     spons (2.24e-03)      recon (2.99e-03)         ín (2.61e-03)   
11       (!! (2.24e-03)      Licht (2.64e-03)        cot (2.46e-03)   
12     feliz (2.24e-03)        apo (2.55e-03)        ños (2.24e-03)   
13     lesen (2.24e-03)        boy (2.40e-03)       aved (2.11e-03)   
14   kontrol (1.75e-03)      ombok (2.26e-03)        får (2.03e-03)   
15      helf (1.64e-03)         lj (2.12e-03)     kommun (1.97e-03)   
16    spiele (1.54e-03)         co (2.04e-03)       così (1.91e-03)   
17       )": (1.54e-03)       alus (1.98e-03)      etten (1.91e-03)   
18       fas (1.45e-03)        rus (1.98e-03)       rose (1.74e-03)   
19      -git (1.45e-03)        tat (1.93e-03)       cron (1.74e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.5703)     contador (0.8398)         , (0.6211)   
1       and (0.1436)         bö (7.26e-03)       and (0.1367)   
2       the (0.1377)   karakter (7.26e-03)       the (0.1011)   
3        in (0.0591)    kontrol (7.26e-03)       ' ' (0.0554)   
4       ' ' (0.0552)         �� (5.65e-03)        in (0.0537)   
5         a (0.0132)         �� (5.65e-03)         a (0.0119)   
6       ( (3.66e-03)      KANJI (3.43e-03)       ( (4.67e-03)   
7      to (3.39e-03)      subur (3.43e-03)      to (2.96e-03)   
8      of (2.90e-03)       samt (3.43e-03)      of (2.61e-03)   
9      by (2.62e-03)     testim (2.67e-03)     

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [14]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0260)         .vn (0.0199)       /entities (0.0279)   
1         /problem (0.0140)   /Register (0.0114)        /problem (0.0149)   
2      /problems (9.27e-03)    testim (6.92e-03)     /problems (9.91e-03)   
3        /global (6.76e-03)      sagt (6.53e-03)       /global (6.91e-03)   
4   /environment (6.08e-03)     asign (5.23e-03)     /provider (6.28e-03)   
5      /provider (5.89e-03)       -ie (4.93e-03)  /environment (5.99e-03)   
6         .Today (5.88e-03)     zeigt (4.07e-03)   /connection (5.98e-03)   
7    /connection (5.69e-03)        że (3.93e-03)        .Today (5.46e-03)   
8        /manage (5.62e-03)      -ves (3.30e-03)     /customer (4.92e-03)   
9      /customer (4.80e-03)   personn (2.84e-03)       /manage (4.88e-03)   
10  /preferences (4.26e-03)         ť (2.84e-03)  /preferences (4.04e-03)   
11       /shared (3.36e-03)     probs (2.78e-03)       /shared (3.41e-03)   
12       /dialog (3.30e-03)      elig (2.55e-03)       /dialog (3.39e-03)   
13      /account (3.17e-03)    ):\n\n (2.39e-03)      /account (3.37e-03)   
14       /entity (3.15e-03)      roku (2.32e-03)      libertin (3.25e-03)   
15      libertin (3.01e-03)     lesen (2.23e-03)       /entity (3.20e-03)   
16       /layout (2.87e-03)       )": (2.22e-03)      /effects (3.05e-03)   
17          .Try (2.85e-03)  ,,,,,,,, (2.20e-03)       /layout (3.02e-03)   
18      /effects (2.72e-03)    spiele (2.12e-03)          .Try (2.90e-03)   
19        /legal (2.67e-03)       (us (2.09e-03)        /legal (2.67e-03)   

                                                                    \
                 ft_inv                  diff             diff_inv   
0          .vn (0.0215)         acea (0.0416)     nails (5.15e-03)   
1    /Register (0.0111)         ucci (0.0156)       row (4.27e-03)   
2     testim (6.69e-03)         anno (0.0104)     calle (3.93e-03)   
3       sagt (6.56e-03)      Blond (6.98e-03)      ropa (3.74e-03)   
4        -ie (5.00e-03)       heim (6.19e-03)      emos (3.68e-03)   
5      asign (4.94e-03)        lij (5.33e-03)       cho (3.31e-03)   
6      zeigt (4.50e-03)        ple (5.08e-03)      yles (3.21e-03)   
7         że (3.76e-03)     ervers (3.37e-03)   Bedford (2.91e-03)   
8       -ves (3.31e-03)     ighted (2.87e-03)        ー� (2.68e-03)   
9          ť (3.25e-03)      avras (2.32e-03)       rem (2.56e-03)   
10     probs (2.87e-03)       ając (2.32e-03)        Ps (1.97e-03)   
11   personn (2.77e-03)     itives (2.26e-03)      nail (1.95e-03)   
12      roku (2.36e-03)       orda (2.21e-03)      ason (1.92e-03)   
13      elig (2.33e-03)        Lah (2.13e-03)     venta (1.91e-03)   
14     lesen (2.29e-03)      Arial (1.94e-03)    olicit (1.77e-03)   
15  ,,,,,,,, (2.28e-03)     aturas (1.72e-03)   lements (1.73e-03)   
16    ):\n\n (2.21e-03)      annon (1.64e-03)         ś (1.73e-03)   
17       )": (2.20e-03)   independ (1.56e-03)     ISSUE (1.62e-03)   
18       esl (2.00e-03)      pelic (1.56e-03)       arp (1.56e-03)   
19    spiele (1.96e-03)   markdown (1.55e-03)    olland (1.54e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.7981)     contador (0.9628)         , (0.8091)   
1        ' ' (0.1089)    kontrol (3.26e-03)       ' ' (0.1180)   
2        the (0.0435)   karakter (2.43e-03)       the (0.0329)   
3        and (0.0315)       rekl (1.61e-03)       and (0.0243)   
4       in (6.12e-03)         bö (1.46e-03)       ( (5.48e-03)   
5        ( (4.51e-03)       zoek (1.15e-03)      in (4.46e-03)   
6       's (2.70e-03)    Produto (9.57e-04)      's (2.14e-03)   
7        a (1.69e-03)     testim (9.23e-04)       a (1.20e-03)   
8       to (1.09e-03)     bilder (8.02e-04)      to (8.62e-04)   
9        . (6.79e-04)         �� (7.68e-04)       . (6.71e

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [15]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                           \
                       base                       ft   
0           .Today (0.0254)          .Today (0.0226)   
1        .Second (9.16e-03)       .Second (8.77e-03)   
2        Buccane (6.49e-03)       Buccane (6.48e-03)   
3        /Area (5.90e-03) ✅       /Area (6.08e-03) ✅   
4            .au (5.27e-03)           .au (4.31e-03)   
5            aru (5.11e-03)       /Math (3.26e-03) ✅   
6    /problems (3.10e-03) ✅   /problems (3.06e-03) ✅   
7           fter (2.74e-03)          ilot (2.81e-03)   
8     confidence (2.71e-03)           aru (2.70e-03)   
9           ilot (2.59e-03)    confidence (2.70e-03)   
10       /Math (2.54e-03) ✅          fter (2.51e-03)   
11   /entities (2.44e-03) ✅   /activity (2.49e-03) ✅   
12    /problem (2.18e-03) ✅       /bind (2.46e-03) ✅   
13   /activity (2.09e-03) ✅   /entities (2.39e-03) ✅   
14       /bind (2.09e-03) ✅          oire (2.31e-03)   
15   persistence (1.92e-03)    /problem (2.28e-03) ✅   
16    /respond (1.90e-03) ✅   persistence (2.11e-03)   
17     belonging (1.90e-03)    /respond (1.86e-03) ✅   
18   /operator (1.84e-03) ✅   /operator (1.80e-03) ✅   
19  /community (1.57e-03) ✅     belonging (1.74e-03)   

                                          layer_14                        \
                        diff                  base                    ft   
0             ' ' (8.53e-03)           To (0.7201)           To (0.7643)   
1    coordinate (6.70e-03) ✅          ### (0.1250)           ** (0.1125)   
2           might (5.75e-03)           ** (0.0762)          ### (0.0682)   
3      synchron (4.34e-03) ✅        Let (0.0591) ✅        Let (0.0381) ✅   
4          '\n\n' (4.30e-03)          The (0.0132)        The (8.85e-03)   
5           encia (3.91e-03)  Certainly (1.52e-03)  Certainly (2.75e-03)   
6           Mon (3.76e-03) ✅       Sure (1.08e-03)       Sure (1.74e-03)   
7            Than (3.70e-03)         ## (7.44e-04)         In (6.42e-04)   
8          reader (3.61e-03)         In (5.11e-04)         ## (5.90e-04)   
9     financial (3.32e-03) ✅    Given (2.27e-04) ✅     Here (3.66e-04) ✅   
10       target (3.21e-03) ✅    First (2.23e-04) ✅    Given (2.46e-04) ✅   
11          items (3.07e-03)    Alright (1.27e-04)    First (1.96e-04) ✅   
12          super (2.80e-03)       We (1.14e-04) ✅       This (1.34e-04)   
13     deadline (2.67e-03) ✅        1 (1.07e-04) ✅        For (1.14e-04)   
14      morning (2.47e-03) ✅       #### (1.01e-04)         As (1.12e-04)   
15           '  ' (2.44e-03)        ``` (9.49e-05)    Alright (1.12e-04)   
16              1 (2.34e-03)     Here (8.92e-05) ✅        1 (8.85e-05) ✅   
17            ... (2.32e-03)     This (8.92e-05) ✅       #### (7.34e-05)   
18          supra (2.28e-03)        For (5.39e-05)         We (6.48e-05)   
19        Picture (2.10e-03)         As (5.28e-05)         It (5.84e-05)   

                                          layer_15                        \
                        diff                  base                    ft   
0               You (0.1748)           To (0.4297)           To (0.4785)   
1              Your (0.1576)          ### (0.2598)           ** (0.3730)   
2               You (0.1481)           ** (0.2598)          ### (0.1069)   
3             you (0.1335) ✅        Let (0.0242) ✅        Let (0.0145) ✅   
4            your (0.0809) ✅          The (0.0166)          The (0.0128)   
5        yourself (0.0480) ✅  Certainly (2.55e-03)  Certainly (4.15e-03)   
6              Your (0.0480)       Sure (1.37e-03)       Sure (2.21e-03)   
7            your (0.0163) ✅         In (1.21e-03)         In (1.73e-03)   
8            here (9.68e-03)         ## (8.28e-04)         ## (5.61e-04)   
9            YOUR (7.38e-03)    Given (3.91e-04) ✅       Here (4.37e-04)   
10            YOU (7.38e-03)    First (2.37e-04) ✅          A (4.37e-04)   
11           YOUR (5.87e-03)        1 (2.37e-04) ✅    Given (3.85e-04) ✅   
12            YOU (5.50e-03)          A (1.96e-04)

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [16]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0          problem (0.0281) ✅                -> (0.0622)   
1         /problem (0.0264) ✅                's (0.0449)   
2                the (0.0251)            '\n\n' (0.0341)   
3                 's (0.0249)         problem (0.0333) ✅   
4            solve (0.0239) ✅             :\n\n (0.0310)   
5              :\n\n (0.0231)               you (0.0197)   
6                 -> (0.0206)        /problem (0.0178) ✅   
7        /entities (0.0163) ✅              '\n' (0.0169)   
8        /problems (0.0139) ✅               the (0.0160)   
9              you (9.01e-03)           solve (0.0138) ✅   
10       /manage (8.69e-03) ✅       /entities (0.0116) ✅   
11            this (7.87e-03)       /problems (0.0112) ✅   
12             :\n (6.59e-03)                 , (0.0110)   
13          '\n\n' (6.42e-03)             :\n (9.68e-03)   
14    understand (6.40e-03) ✅         seems (7.46e-03) ✅   
15           seems (5.59e-03)            this (6.57e-03)   
16               , (4.45e-03)     statement (6.41e-03) ✅   
17       analyze (4.31e-03) ✅    understand (6.00e-03) ✅   
18        .Today (4.23e-03) ✅       /manage (5.81e-03) ✅   
19       address (3.80e-03) ✅              is (5.63e-03)   
20       /global (3.59e-03) ✅              ’s (4.58e-03)   
21  /preferences (3.58e-03) ✅            your (4.57e-03)   
22        solves (3.06e-03) ✅               : (4.41e-03)   
23            '\n' (3.03e-03)      problems (3.78e-03) ✅   
24            your (2.83e-03)       address (3.72e-03) ✅   
25      question (2.66e-03) ✅      question (3.55e-03) ✅   
26        tackle (2.65e-03) ✅        .Today (2.78e-03) ✅   
27          /job (2.55e-03) ✅       /global (2.72e-03) ✅   
28          math (2.46e-03) ✅        solves (2.67e-03) ✅   
29      problems (2.37e-03) ✅        puzzle (2.48e-03) ✅   
30     /provider (2.29e-03) ✅          math (2.39e-03) ✅   
31        puzzle (2.19e-03) ✅      requires (2.35e-03) ✅   
32              ’s (2.19e-03)      involves (2.30e-03) ✅   
33         break (2.06e-03) ✅           .\n\n (1.95e-03)   
34       /crypto (2.02e-03) ✅  /preferences (1.95e-03) ✅   
35       /layout (1.85e-03) ✅     /provider (1.79e-03) ✅   
36      /logging (1.73e-03) ✅       /crypto (1.78e-03) ✅   
37      /effects (1.66e-03) ✅       puzzles (1.59e-03) ✅   
38               : (1.64e-03)   /connection (1.56e-03) ✅   
39   /connection (1.62e-03) ✅        tackle (1.50e-03) ✅   
40        decode (1.50e-03) ✅          step (1.40e-03) ✅   
41     statement (1.44e-03) ✅       /engine (1.31e-03) ✅   
42              we (1.35e-03)       /layout (1.25e-03) ✅   
43       /engine (1.35e-03) ✅      /logging (1.22e-03) ✅   
44       /object (1.34e-03) ✅      solution (1.15e-03) ✅   
45       puzzles (1.24e-03) ✅       /object (1.15e-03) ✅   
46          step (1.16e-03) ✅      /effects (1.07e-03) ✅   
47  /environment (1.13e-03) ✅       /shared (1.06e-03) ✅   
48       /entity (1.04e-03) ✅             ' ' (1.05e-03)   
49       /shared (1.03e-03) ✅       analyze (1.04e-03) ✅   
50  /application (1.02e-03) ✅          /job (1.02e-03) ✅   
51          begins (1.01e-03)     /activity (9.60e-04) ✅   
52     /activity (1.00e-03) ✅      /company (9.48e-04) ✅   
53        /legal (9.81e-04) ✅      exercise (9.26e-04) ✅   
54          word (8.54e-04) ✅         break (8.87e-04) ✅   
55      exercise (8.50e-04) ✅              in (8.58e-04)   
56          task (7.89e-04) ✅        answer (8.21e-04) ✅   
57             for (7.83e-04)  /environment (6.78e-04) ✅   
58        answer (7.77e-04) ✅       appears (6.72e-04) ✅   
59           /pr (7.66e-04) ✅         ' \n\n' (6.68e-04)   
60          /con (6.63e-04) ✅           /pr (6.51e-04) ✅   
61          /reg (6.18e-04) ✅        begins (6.39e-04) ✅   
62      /testing (6.10e-04) ✅      /testing (6.31e-04) ✅   
63      WHATSOEVER (5.80e-04)          /reg (6.01e-04) ✅   
64        /tasks (5.39e-04) ✅          /con (5.93e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [17]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                \
                  pos_-3                 pos_-1                  pos_0   
0          Sink (0.0304)      gorith (9.77e-03)         imo (9.40e-03)   
1    caratter (9.28e-03)       encia (7.17e-03)        agra (7.35e-03)   
2        kins (7.66e-03)        onta (6.71e-03)        rate (7.35e-03)   
3          el (7.20e-03)     trovare (5.58e-03)         ate (5.37e-03)   
4        pard (7.20e-03)   -scalable (4.09e-03)        ates (5.04e-03)   
5        elin (5.98e-03)          }? (4.09e-03)    ategorie (4.46e-03)   
6       orama (4.94e-03)   acimiento (3.60e-03)        ucci (3.94e-03)   
7         apo (4.12e-03)     Masters (3.17e-03)       avras (3.94e-03)   
8       elian (3.86e-03)    Accounts (2.81e-03)     Opening (3.46e-03)   
9         alu (3.63e-03)      stride (2.81e-03)          Fe (3.25e-03)   
10  selection (3.01e-03)         sab (2.81e-03)   slightest (3.07e-03)   
11   enslaved (3.01e-03)        .;\n (2.64e-03)        |,\n (3.07e-03)   
12        oll (2.82e-03)       abbix (2.47e-03)        reib (3.07e-03)   
13         ån (2.82e-03)  Executable (2.47e-03)        bill (2.87e-03)   
14        Loc (2.66e-03)       guard (2.47e-03)         tat (2.87e-03)   
15        loi (2.49e-03)        ucci (2.32e-03)     aproxim (2.87e-03)   
16          片 (2.20e-03)        (SQL (2.18e-03)    independ (2.87e-03)   
17      posto (2.20e-03)      resden (2.18e-03)        orsi (2.38e-03)   
18       otec (2.20e-03)          si (2.18e-03)         iji (2.38e-03)   
19     risult (2.06e-03)        atra (2.04e-03)        argv (2.24e-03)   

                                                                          \
                    pos_1                 pos_2                    pos_3   
0          avras (0.0126)         heim (0.0164)            acea (0.0302)   
1          ate (7.17e-03)         acea (0.0120)            anno (0.0172)   
2         orsi (6.32e-03)         anno (0.0120)            ucci (0.0111)   
3         bags (5.95e-03)         ucci (0.0106)          heim (7.63e-03)   
4      surplus (5.58e-03)       lore (8.24e-03)         pelic (6.71e-03)   
5        Trail (5.25e-03)      pelic (8.24e-03)      independ (5.92e-03)   
6         ates (4.33e-03)   independ (5.65e-03)          lore (4.91e-03)   
7       stride (4.21e-03)        tat (4.12e-03)            co (4.33e-03)   
8         ..." (4.21e-03)      avras (3.65e-03)           tat (4.09e-03)   
9         rate (4.09e-03)       Pert (3.65e-03)         ativa (3.83e-03)   
10         owi (3.60e-03)       rate (3.42e-03)           apo (2.98e-03)   
11        heim (3.39e-03)    surplus (3.42e-03)          Pert (2.88e-03)   
12          ta (3.39e-03)    isateur (2.58e-03)      antanamo (2.81e-03)   
13         apo (3.28e-03)        ily (2.58e-03)   recognition (2.81e-03)   
14        ucci (3.28e-03)      annon (2.50e-03)           OCC (2.81e-03)   
15    antanamo (2.90e-03)         lj (2.27e-03)            lj (2.32e-03)   
16        Pert (2.90e-03)       ates (2.27e-03)           995 (2.18e-03)   
17       posed (2.72e-03)        ate (2.27e-03)           rus (1.87e-03)   
18   slightest (2.72e-03)         Zu (2.21e-03)         ombok (1.87e-03)   
19        onta (2.64e-03)        lij (1.89e-03)           boy (1.65e-03)   

                                                                     \
                  pos_10               pos_50               pos_100   
0          anno (0.0562)        acea (0.0417)         acea (0.0496)   
1          ucci (0.0206)        ucci (0.0154)         ucci (0.0125)   
2          heim (0.0125)     Blond (8.24e-03)      Blond (9.16e-03)   
3        acea (5.92e-03)      anno (7.26e-03)        lij (5.55e-03)   
4        Pert (4.06e-03)      heim (6.41e-03)       heim (4.91e-03)   
5          co (3.81e-03)       ple (6.01e-03)       anno (4.06e-03)   
6         apo (3.59e-03)       lij (5.65e-03)        ple (4.06e-03)   
7       annon (3.59e-03)    ervers (5.31e-03)     ighted (4.06e-03)   
8    

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [18]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                            \
                        pos_-3                    pos_-1   
0              Edited (0.0157)            ' ' (8.53e-03)   
1                 alu (0.0134)   coordinate (6.70e-03) ✅   
2             atory (9.42e-03)          might (5.75e-03)   
3                ih (8.57e-03)     synchron (4.34e-03) ✅   
4              haft (6.49e-03)         '\n\n' (4.30e-03)   
5               que (4.65e-03)          encia (3.91e-03)   
6              Sink (4.52e-03)          Mon (3.76e-03) ✅   
7            laws (4.32e-03) ✅           Than (3.70e-03)   
8        statutes (4.23e-03) ✅         reader (3.61e-03)   
9                og (3.69e-03)    financial (3.32e-03) ✅   
10              ate (3.59e-03)       target (3.21e-03) ✅   
11    Legislation (2.91e-03) ✅          items (3.07e-03)   
12        character (2.83e-03)          super (2.80e-03)   
13             Meer (2.75e-03)     deadline (2.67e-03) ✅   
14        selection (2.70e-03)      morning (2.47e-03) ✅   
15             Hier (2.69e-03)           '  ' (2.44e-03)   
16   AttributeError (2.48e-03)              1 (2.34e-03)   
17            lég (2.33e-03) ✅            ... (2.32e-03)   
18              ell (2.21e-03)          supra (2.28e-03)   
19              AZE (2.20e-03)        Picture (2.10e-03)   

                                                                             \
                     pos_0                   pos_1                    pos_2   
0           ann (0.1003) ✅          dog (0.0465) ✅              -> (0.0348)   
1           man (0.0690) ✅            ' ' (0.0349)       chicken (0.0137) ✅   
2            blue (0.0609)      chicken (0.0197) ✅      shrimp (8.53e-03) ✅   
3              -> (0.0561)           '\n' (0.0170)             " (8.01e-03)   
4             ' ' (0.0403)         person (0.0169)           -ch (7.23e-03)   
5         hello (0.0238) ✅            man (0.0163)       sauce (6.66e-03) ✅   
6               > (0.0201)          bee (0.0151) ✅        gloves (5.94e-03)   
7             MAN (0.0136)              - (0.0129)        fish (5.41e-03) ✅   
8            -> (7.97e-03)       turtle (0.0126) ✅             : (5.08e-03)   
9             / (6.84e-03)       monkey (0.0108) ✅            ch (4.58e-03)   
10     friend (6.84e-03) ✅        cat (9.63e-03) ✅           mer (4.54e-03)   
11            " (6.50e-03)            : (8.91e-03)     dinners (4.52e-03) ✅   
12        bye (4.79e-03) ✅         blue (8.87e-03)        food (4.48e-03) ✅   
13            _ (4.65e-03)           -> (8.68e-03)          does (4.15e-03)   
14            ~ (4.58e-03)          guy (5.53e-03)   chocolate (3.86e-03) ✅   
15         bear (3.92e-03)       '\n\n' (5.24e-03)             > (3.78e-03)   
16   stranger (3.77e-03) ✅         last (5.13e-03)        soup (3.51e-03) ✅   
17          HEL (3.46e-03)        ape (4.85e-03) ✅     addresses (3.39e-03)   
18            > (3.45e-03)      chick (4.80e-03) ✅            to (3.34e-03)   
19         test (3.33e-03)   squirrel (4.70e-03) ✅      dishes (3.34e-03) ✅   

                                        layer_14                            \
                    pos_3                 pos_-3                    pos_-1   
0             -> (0.0498)        inne (0.0387) ✅              You (0.1748)   
1            ' ' (0.0405)        kurs (0.0191) ✅             Your (0.1576)   
2         blue (0.0176) ✅        mies (0.0107) ✅              You (0.1481)   
3            ice (0.0117)      konk (8.44e-03) ✅            you (0.1335) ✅   
4            " (8.50e-03)      indy (7.35e-03) ✅           your (0.0809) ✅   
5        DNA (8.23e-03) ✅        zg (6.95e-03) ✅       yourself (0.0480) ✅   
6            : (8.07e-03)      spos (6.61e-03) ✅             Your (0.0480)   
7           is (7.40e-03)      sond (6.35e-03) ✅           your (0.0163) ✅   
8            - (6.26e-03)        )!\n (4.87e-03)           here (9.68e-03)   
9            > (5.14e-03)      darm (4.37e-03) ✅           YOUR (7.38e-03)   
10   chicken (4.96e-03) ✅  